# Full-Arm Manipulation Planner

Loads the full Panda arm + hand + bottle-cap + obstacle scene (same setup as
`full_arm_nhr.ipynb`), generates a C-free cover with Drake's **native**
IRIS-ZO / clique-cover implementation (unmodified - no custom Python
reimplementation from #69-71), and plans a trajectory with `ManipulationPlanner`,
using the grasp polytope derived from the wrist-axis analysis (#80, see
`algorithms/nlp_sampling/tc_space_investigation_summary.md` Part 2) instead of
a hand-picked box.

This is a first-pass scaffold: sections 1-2 have been run and validated
end-to-end; section 4's IRIS-ZO clique-cover + MOSEK MIQP solve is expensive
(minutes+, needs a MOSEK license) and has **not** been executed as part of
writing this notebook - see the caveats in section 4 before running it.

In [6]:
# Cell 1: Import necessary libraries

import os
import numpy as np
import time
from pydrake.all import (
    DiagramBuilder, AddMultibodyPlantSceneGraph, Parser,
    Role, MeshcatVisualizer, StartMeshcat, RationalForwardKinematics, CspaceFreePolytope,
    SeparatingPlaneOrder, Rgba, RollPitchYaw,
    LinearEqualityConstraint, Sphere, Parallelism, AddDefaultVisualization,
    ConnectPlanarSceneGraphVisualizer, IrisFromCliqueCoverOptions,
    CommonSampledIrisOptions, IrisZoOptions, IrisNp2Options,
    IrisInConfigurationSpaceFromCliqueCover, RandomGenerator, RobotDiagramBuilder,
    SceneGraphCollisionChecker, MultibodyPlant, SceneGraph,
    SolverOptions, CommonSolverOption, GeometrySet, ScsSolver
)
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.geometry.optimization import ConvexHull as DrakeConvexHull
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.trajectories import CompositeTrajectory
from pydrake.common import FindResourceOrThrow
from scipy.spatial import ConvexHull
from pydrake.math import RigidTransform, RotationMatrix
from pydrake.multibody.inverse_kinematics import InverseKinematics
from pydrake.solvers import Solve, SnoptSolver, IpoptSolver, OsqpSolver, ScsSolver
from functools import partial
from dataclasses import dataclass
import matplotlib.pyplot as plt
from ipywidgets import widgets

from pathlib import Path
import sys

def _find_repo_root(start: Path, markers=(".git", "README.md")) -> Path:
    for candidate in (start, *start.parents):
        if any((candidate / m).exists() for m in markers):
            return candidate
    raise RuntimeError(f"Could not locate repo root above {start}")

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (REPO_ROOT, REPO_ROOT / "algorithms", REPO_ROOT / "algorithms" / "nlp_sampling"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

from ciris_plant_visualizer import CIrisPlantVisualizer

## 1. Load the full Panda arm + cap + obstacles

Verbatim from `full_arm_nhr.ipynb` (same URDFs, same welds, same obstacle placement) - this is what makes the wrist-axis polytope built in section 2 (from a *separate*, obstacle-free plant in `wrist_axis_grasp_polytope.py`) valid here: same URDF load order means the same position indexing.

In [7]:
print("Setting up the plant and scene graph...")

builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

Setting up the plant and scene graph...


In [8]:
print("Loading Panda robot models...")

# panda_arm  = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_arm.urdf")[0]
panda_arm = parser.AddModels(url="file:///home/julialopezgomez/optimisation-based-manipulation-planner/my_sdfs/panda_arm.urdf")[0]
panda_hand = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]

print("Welding Panda arm and hand...")

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("panda_link0", panda_arm),
    RigidTransform())

X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(
    plant.GetFrameByName("panda_link8", panda_arm),
    plant.GetFrameByName("panda_hand", panda_hand),
    X_8H)

print("Setting default finger joint positions...")
plant.GetJointByName("panda_finger_joint1", panda_hand).set_default_translation(-0.024)

Loading Panda robot models...
Welding Panda arm and hand...
Setting default finger joint positions...


In [9]:
print("Adding the bottle cap and obstacles...")

cap = parser.AddModels(file_name="my_sdfs/bottle_cap.sdf")[0]
obstacle1 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
obstacle3 = parser.AddModels("my_sdfs/obstacle.sdf")[0]

plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("base_link", cap),
    RigidTransform(RotationMatrix(), [0.5, 0, 0]))

obstacle_pose1 = RigidTransform(RotationMatrix(), [0.51, 0.031, 0.01])
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle1),
    obstacle_pose1)

obstacle_pose3 = RigidTransform(RotationMatrix(), [0.467, -0.005, 0.01])
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle3),
    obstacle_pose3)

Adding the bottle cap and obstacles...


<WeldJoint name='world_welds_to_obstacle_link' index=16 model_instance=6>

In [10]:
print("Finalising plant...")
print("Bodies:", plant.num_bodies(), "Joints:", plant.num_joints())


plant.Finalize()

inspector = scene_graph.model_inspector()
num_frames = len(list(inspector.GetAllFrameIds()))
num_geoms = sum(inspector.NumGeometriesForFrame(fid) for fid in inspector.GetAllFrameIds())
print("Frames:", num_frames, "Geometries:", num_geoms)


print("Plant finalised.")

print("Number of positions: ", plant.num_positions())


q_star = np.zeros(plant.num_positions())

print("Initialising CspaceFreePolytope...")
cspace_free_polytope = CspaceFreePolytope(
    plant,
    scene_graph,
    SeparatingPlaneOrder.kAffine,
    q_star)

print("Initializing CIrisPlantVisualizer...")
visualizer = CIrisPlantVisualizer(
    plant,
    builder,
    scene_graph,
    cspace_free_polytope,
    viz_role=Role.kIllustration,
    allow_plus_3dof=True
)

visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)

INFO:drake:Meshcat listening for connections at http://localhost:7002


Finalising plant...
Bodies: 18 Joints: 17
Frames: 18 Geometries: 94
Plant finalised.
Number of positions:  10
Initialising CspaceFreePolytope...
Initializing CIrisPlantVisualizer...
Visualisations won't work properly. Can't visualize the TC-Space of plants with more than 3-DOF. The first 3 DOF from the plant will be visualized


In [11]:
sliders = []

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]
    q_high = plant.GetPositionUpperLimits()[i]
    step = (q_high - q_low) / 100
    sliders.append(widgets.FloatSlider(
        min=q_low, max=q_high, 
        value=0, step=step, 
        description=f"q{i}"))
    
q = np.zeros(plant.num_positions())

def report_collisions(q):
    plant.SetPositions(plant_context, q)
    query_object = scene_graph.get_query_output_port().Eval(
        scene_graph.GetMyContextFromRoot(diagram_context)
    )
    inspector = scene_graph.model_inspector()

    pairs = query_object.ComputePointPairPenetration()
    if not pairs:
        print("No penetrations")
        return

    for pair in pairs:
        name_A = inspector.GetName(pair.id_A)
        name_B = inspector.GetName(pair.id_B)
        print(f"{name_A} <-> {name_B}, depth={pair.depth}")



out = widgets.Output()
display(out)

def handle_slider_change(change, idx):
    q[idx] = change["new"]
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    with out:
        out.clear_output(wait=True)
        
        report_collisions(q)
        print(f"{visualizer.check_collision_q_by_ik(q)} \t {q}", flush=True)
    
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

Output()

FloatSlider(value=0.0, description='q0', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q1', max=1.7628, min=-1.7628, step=0.035255999999999996)

FloatSlider(value=0.0, description='q2', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=-0.0698, description='q3', max=-0.0698, min=-3.0718, step=0.03002)

...

FloatSlider(value=0.0, description='q5', max=3.7525, min=-0.0175, step=0.0377)

FloatSlider(value=0.0, description='q6', max=1.0, min=-1.0, step=0.02)

FloatSlider(value=0.0, description='q7', max=0.0, min=-0.045, step=0.00045)

FloatSlider(value=0.0, description='q8', max=0.045, step=0.00045)

FloatSlider(value=0.0, description='q9', max=3.14, min=-3.14, step=0.06280000000000001)

In [43]:
def joint_position_index(plant, joint_name, model_instance=None):
    if model_instance is None:
        joint = plant.GetJointByName(joint_name)
    else:
        joint = plant.GetJointByName(joint_name, model_instance)

    if joint.num_positions() != 1:
        raise ValueError(
            f"Joint {joint_name} has {joint.num_positions()} positions; "
            "this helper assumes a 1-DOF joint."
        )

    return joint.position_start()

for i, name in enumerate(plant.GetPositionNames()):
    print(f"{i:2d}: {name}")

 0: panda_panda_joint1_q
 1: panda_panda_joint2_q
 2: panda_panda_joint3_q
 3: panda_panda_joint4_q
 4: panda_panda_joint5_q
 5: panda_panda_joint6_q
 6: panda_panda_joint7_q
 7: panda_hand_panda_finger_joint1_x
 8: panda_hand_panda_finger_joint2_x
 9: bottle_cap_cap_to_base_q


## 2. Grasp polytope from the wrist-axis analysis (#80)

Reuses `wrist_axis_grasp_polytope.py`'s already-validated construction directly
(see `tc_space_investigation_summary.md` Part 2 for the full derivation and
plots): the wrist joint's rotation axis passes through the end-effector
frame's own origin, so the grasp constraint is *exactly* (not approximately)
invariant to it - the feasible region is the wrist's full range times a
small `+-0.001` rad margin on the other 6 arm joints, fingers pinned at
their found grasp width.

That construction runs on its own separate, obstacle-free plant
(`nlp_sampling_standalone_test.build_minimal_plant()`), built from the same
URDFs in the same order as section 1's plant - the index-parity asserts
below confirm the two plants agree on position ordering before any array
built on one is reused on the other.

Composed here into two full 10-dim boxes over *this* plant's actual
configuration space (`ManipulationPlanner` needs `CP`/`CG` in the same
ambient dimension as the `cs_free` regions IRIS-ZO will generate in
section 4, not just the 4 "gripper" dims):

- **`grasp_polytope`** - the wrist-axis box on the 6 arm joints + wrist,
  the fingers' own physical grasp tolerance (`[-0.025,-0.024]`/
  `[0.024,0.025]`, the same constants used everywhere else in this project -
  not q0-specific), and the cap left **free** over its full range. The cap
  can't be pinned here: it's the DOF that actually advances between
  grasp/release cycles in the MIQP below (`ManipulationPlanner`'s
  constraints 3/4 tie *consecutive* grasp/release cap values together, not
  every point to one fixed value) - pinning it would force every grasp/
  release knot point to the same cap angle and make the problem trivial or
  infeasible.
- **`placement_polytope`** - the same convention `full_arm_nhr.ipynb` used
  (wrist full range, a wider/more-open finger range, cap free), lifted from
  4D to the full 10D space by leaving the 6 arm joints unconstrained
  (placement was never about a specific arm posture, only the gripper
  end-state). By construction `grasp_polytope` is a strict subset of
  `placement_polytope` here.

In [61]:
from wrist_axis_grasp_polytope import (
    ARM_JOINT_INDICES, build_constraints as _build_standalone_constraints,
    find_grasp_seed as _find_standalone_grasp_seed, build_simple_margin_polytope,
)
from nlp_sampling_standalone_test import make_joint_indices

GRASP_MARGIN = 0.0005  # matches the finger joints' own grasp tolerance width - see
                       # tc_space_investigation_summary.md Part 2 for the full derivation.

_standalone_constraints = _build_standalone_constraints()
q0 = _find_standalone_grasp_seed(_standalone_constraints)
lower_wrist_box, upper_wrist_box = build_simple_margin_polytope(_standalone_constraints, q0, GRASP_MARGIN)

idx = make_joint_indices(plant, panda_arm, panda_hand, cap)
idx_wrist, idx_f1, idx_f2, idx_cap = idx["idx_wrist"], idx["idx_f1"], idx["idx_f2"], idx["idx_cap"]

print(f"Grasp seed q0 = {np.round(q0, 4)}")
print(f"h(q0) = {_standalone_constraints['h'](q0)}, g(q0) = {_standalone_constraints['g'](q0)}")

# index-parity: the standalone plant (wrist_axis_grasp_polytope.py) and this notebook's
# plant must agree on position ordering before q0/lower_wrist_box/upper_wrist_box (built
# on the standalone plant) can be reused directly here.
assert list(idx["idx_arm"][:6]) == ARM_JOINT_INDICES
assert idx_wrist == _standalone_constraints["idx_wrist"] == 6
assert (idx_f1, idx_f2, idx_cap) == (7, 8, 9)
print("index-parity assertions OK")

Grasp seed q0 = [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708   0.9944 -0.024   0.0247
  0.9449]
h(q0) = [-1.28691283e-05 -1.01563271e-04 -4.77217183e-04 -9.20447251e-05], g(q0) = [ 0.0009401  -0.0059401  -0.09999988]
index-parity assertions OK


In [62]:
plant.SetPositions(plant_context, q0)
diagram.ForcedPublish(diagram_context)



In [55]:
grasp_lower, grasp_upper = lower_wrist_box.copy(), upper_wrist_box.copy()
grasp_lower[idx_f1], grasp_upper[idx_f1] = -0.025, -0.024
grasp_lower[idx_f2], grasp_upper[idx_f2] = 0.024, 0.025
grasp_lower[idx_cap] = plant.GetPositionLowerLimits()[idx_cap]
grasp_upper[idx_cap] = plant.GetPositionUpperLimits()[idx_cap]
grasp_polytope = HPolyhedron.MakeBox(grasp_lower, grasp_upper)

placement_lower = plant.GetPositionLowerLimits().copy()
placement_upper = plant.GetPositionUpperLimits().copy()
placement_lower[idx_wrist], placement_upper[idx_wrist] = -1.0, 1.0
placement_lower[idx_f1], placement_upper[idx_f1] = -0.055, -0.024
placement_lower[idx_f2], placement_upper[idx_f2] = 0.024, 0.055
placement_polytope = HPolyhedron.MakeBox(placement_lower, placement_upper)

print(f"grasp_polytope: ambient_dimension={grasp_polytope.ambient_dimension()}")
print(f"placement_polytope: ambient_dimension={placement_polytope.ambient_dimension()}")
print("q0 in placement:", placement_polytope.PointInSet(q0), " q0 in grasp:", grasp_polytope.PointInSet(q0))

grasp_polytope: ambient_dimension=10
placement_polytope: ambient_dimension=10
q0 in placement: True  q0 in grasp: True


In [60]:
for wrist_value in np.linspace(lower_wrist_box[idx_wrist], upper_wrist_box[idx_wrist], 50):
    q = q0.copy()
    q[idx_wrist] = wrist_value
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    # q within grasping polytope check
    if grasp_polytope.PointInSet(q):
        print(f"q in grasping polytope: {np.round(q, 4)}")
    else:
        print(f"q NOT in grasping polytope: {np.round(q, 4)}")
    time.sleep(0.1)  # slow down the visualization so we can see the motion

q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -1.     -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.9592 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.9184 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.8776 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.8367 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.7959 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.7551 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.7143 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6525  1.8095 -2.2268  1.7824  1.708  -0.6735 -0.024   0.0247
  0.9449]
q in grasping polytope: [-1.7375 -1.6

In [57]:
# Sample grasping polytope and show in meshcat
grasp_samples = []

for i in range(100):
    q_sample = grasp_polytope.UniformSample(
        generator=RandomGenerator(),
        previous_sample=grasp_samples[-1] if grasp_samples else q0
    )
    plant.SetPositions(plant_context, q_sample)
    diagram.ForcedPublish(diagram_context)
    grasp_samples.append(q_sample)
    time.sleep(0.05)


## 3. Manipulation Planner class

Copied verbatim from `algorithms/manipulation_planner/manipulation_planner.py`
(unmodified - same pattern as `manipulation_planner_3dof.ipynb`/
`manipulation_planner_4dof.ipynb`, which inline the class in a cell rather
than importing it). Its `_generate_cfree` is the "IRIS-ZO original Drake
implementation" step: `IrisZoOptions` + `IrisInConfigurationSpaceFromCliqueCover`
straight from `pydrake`, not this project's Python reimplementation from
#69-71.

References `visualizer`, `panda_arm`, `panda_hand`, `cap` as plain globals
(inside `display_trajectory`) - already defined by section 1, so no
module-global wiring is needed here the way the imported version needs
(compare `full_arm_nhr.ipynb`'s `_manipulation_planner_module.visualizer = visualizer` cell).

In [ ]:
def convex_hull_to_hpolyhedron(hull):
    A = hull.equations[:, :-1]
    b = -hull.equations[:, -1]
    return HPolyhedron(A, b)


class ManipulationPlanner():

    def __init__(self,
            visualizer: CIrisPlantVisualizer,
            CP: HPolyhedron,
            CG: HPolyhedron,
            max_grasps: int = 20,
            model_instances: list = None,
            gripper_dim: int = None,
            cs_free: list[HPolyhedron] = None,
            irisalg: str = "irisnp"
        ):
        self.builder = visualizer.builder
        self.plant = visualizer.plant
        self.plant_context = visualizer.plant_context
        self.scene_graph = visualizer.scene_graph
        self.q_star = visualizer.q_star
        self.rat_fk = visualizer.rat_forward_kin
        self.inspector = visualizer.model_inspector
        self.diagram = visualizer.task_space_diagram
        self.diagram_context = visualizer.task_space_diagram_context
        self.lower_joint_limits = visualizer.q_lower_limits
        self.upper_joint_limits = visualizer.q_upper_limits
        self.visualize_cspace = visualizer.visualize_collision_constraint
        self.cspace_free_polytope = visualizer.cspace_free_polytope

        self.CP = CP
        self.CG = CG
        self.max_grasps = max_grasps

        self.model_instances = model_instances if model_instances is not None else [self.plant.GetModelInstanceByName("robot")]

        self.q_dim = self.plant.num_positions()
        self.gripper_dim = gripper_dim if gripper_dim is not None else self.q_dim


        self.cs_free = cs_free if cs_free is not None else self._generate_cfree(irisalg=irisalg)

    def set_max_grasps(self, max_grasps: int):
        self.max_grasps = max_grasps


    def compute_trajectory(self, x_init, x_goal, display=True):
        print("Finding path for minimum grasps")
        path = self._find_minimum_grasp_path(
                x_init = x_init,
                x_goal = x_goal,
                P = self.CP,
                G = self.CG,
                lower_joint_limits = self.lower_joint_limits,
                upper_joint_limits = self.upper_joint_limits,
                c_free_polytopes = self.cs_free
            )

        print("Path found. Generating trajectory")
        traj = self._generate_trajectory(path)


        print("Trajectory generated. Starting display")
        if display:
            self.display_trajectory(traj)

        return path, traj

    def display_trajectory(self, traj, meshcat=True, plotly=False):
        if meshcat:
            num_points = int((traj.end_time() - traj.start_time()) * 4000)
            for t in np.linspace(traj.start_time(), traj.end_time(), num_points):
                q = traj.value(t)
                # self.plant.GetJointByName("left_finger_sliding_joint", gripper).set_translation(self.plant_context, q[2])
                # # self.plant.GetJointByName("right_finger_sliding_joint", gripper).set_translation(self.plant_context, q[3])
                # self.plant.GetJointByName("base_revolute_joint", gripper).set_angle(self.plant_context, q[1])
                # self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[0])
                self.plant.GetJointByName("panda_finger_joint1", panda_hand).set_translation(self.plant_context, q[1][0])
                self.plant.GetJointByName("panda_joint7", panda_arm).set_angle(self.plant_context, q[0][0])
                self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[2][0])
                self.diagram.ForcedPublish(self.diagram_context)
                time.sleep(0.01)
        path = [traj.value(t) for t in np.linspace(traj.start_time(), traj.end_time(), 100)]
        path_q = [q.ravel() for q in path] # Convert to list of numpy arrays
        if plotly:
            self.visualize_cspace(factor=1, num_points=30, paths=[path_q], filled_polytopes=self.cs_free)
        return path_q

    def _generate_trajectory(self, path):
        trajs = []
        print("Generating trajectory from:")
        for q0, q1 in zip(path[:-1], path[1:]):
            print("From\t{}\tto\t{}".format(q0, q1))
            traj, result = self._generate_edge_trajectory(q0, q1)
            if not result.is_success():
                print("failed to generate edge trajectory from {} to {}".format(q0, q1))
                return None
            trajs.append(traj)
        return CompositeTrajectory.AlignAndConcatenate(trajs)


    def _generate_edge_trajectory(self, x_init, x_goal, path_length_weight=None):
        trajopt = GcsTrajectoryOptimization(self.plant.num_positions())
        gcs_regions = trajopt.AddRegions(self.cs_free, order=1, h_min=0.01)
        source = trajopt.AddRegions([Point(x_init)], order=0)
        target = trajopt.AddRegions([Point(x_goal)], order=0)
        trajopt.AddEdges(source, gcs_regions)
        trajopt.AddEdges(gcs_regions, target)
        trajopt.AddPathLengthCost(path_length_weight if path_length_weight is not None else 1.0)
        options = GraphOfConvexSetsOptions()
        [traj, result] = trajopt.SolvePath(source, target, options)
        print(f"result.is_success() = {result.is_success()}")
        print(f"result.get_solution_result() = {result.get_solution_result()}")
        print(f"result.get_solver_id().name() = {result.get_solver_id().name()}")


        # print("\n--- GCS RESULT ---")
        # print("success:", result.is_success())
        # print("solution result:", result.get_solution_result())
        # print("solver:", result.get_solver_id().name())

        # # try:
        # #     details = result.get_solver_details()
        # #     print("solver details:", details)

        # #     for field in [
        # #         "rescode",
        # #         "solution_status",
        # #         "optimizer_time",
        # #     ]:
        # #         if hasattr(details, field):
        # #             print(f"{field}:", getattr(details, field))
        # # except Exception as e:
        # #     print("Could not read solver details:", e)

        # print("------------------\n")

        return traj, result


    def _find_minimum_grasp_path(
            self,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray,
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:

        for i in range(self.max_grasps):
            if i % 10 == 0 and i > 0:
                print(f"Trying with {i} grasps")
            path = self._solve_for_n_grasps_CC(
                i,
                x_init,
                x_goal,
                P,
                G,
                lower_joint_limits,
                upper_joint_limits,
                c_free_polytopes
            )

            if path is not None:
                print(f"Found a solution with {i} grasps")
                return path
        print(f"No solution found for less than {self.max_grasps} grasps")


    def _solve_for_n_grasps_CC(
            self,
            n_grasps: int,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray,
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:
        # Initialize the program
        prog = MathematicalProgram()
        n_points = 2 * n_grasps + 2 # each grasp has a grasp and a release point, plus the initial and goal points
        n_vars = self.q_dim * n_points # for each point of q_dim length, we have n_points*q_dim variables
        non_gripper_dim = self.q_dim - self.gripper_dim # number of non-gripper variables per point

        # Create decision variables
        x = prog.NewContinuousVariables(n_vars, "x")

        # 1. Cost function: Minimize sum of squared distances between consecutive points
        """ ||x_{i+1} - x_{i}||^2  equiv || Cx - c ||^2  == 0 """

        c = np.zeros((n_vars - self.q_dim, 1))
        C = np.zeros((n_vars - self.q_dim, n_vars))
        C[:, self.q_dim:] = np.eye(n_vars - self.q_dim)
        C[:, :-self.q_dim] -= np.eye(n_vars - self.q_dim)

        Q, b = self._to_quadratic_form(C, c)
        Q += np.eye(Q.shape[1]) * 1e-6  # Add a small value to the diagonal to make Q positive definite

        prog.AddQuadraticCost(Q=Q, b=b, vars=x)

        # 2. Initial and goal constraints
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_init.flatten(), x[:self.q_dim])
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_goal.flatten(), x[-self.q_dim:])


        ###### Placement and Grasping Constraints should be adapted task specific #####

        # 3. Placement constraints (equality) -> Cap remains constant in transit paths
        """x_cap_{i+1} - x_cap_{i} == 0 for all even i, odd i+1"""
        for i in range(n_grasps + 1):
            idx = 2*self.q_dim*i # points to first variable of x_i for even i's
            idx1_cap = idx + self.q_dim - 1 # points to last variable of x_i (cap orientation)
            idx2_cap = idx1_cap + self.q_dim # points to last variable of x_{i+1} (cap orientation)
            prog.AddLinearEqualityConstraint(x[idx2_cap] - x[idx1_cap] == 0)

        # 4. Grasp constraints (equality between gripper and cap orientations)
        """x_cap_i - x_wrist_i - x_cap_{i+1} + x_wrist_{i+1} == 0 for odd i, even i+1"""
        for i in range(n_grasps):

            idxp1 = 2*self.q_dim*(i+1) # this idx points to first variable of x_i+1
            idx   = idxp1 - self.q_dim # this idx points to first variable of x_i

            idxp1_wrist = idxp1 + non_gripper_dim
            idxp1_cap = idxp1 + self.q_dim - 1
            idx_wrist = idx + non_gripper_dim
            idx_cap = idx + self.q_dim - 1 # equiv to idxp1 - 1

            prog.AddLinearEqualityConstraint(x[idx_cap] - x[idx_wrist] - x[idxp1_cap] + x[idxp1_wrist] == 0)

        ###### end of manual constraints


        # 5. Inequality Constraints (Placement hull)
        for i in range(n_points):
            prog.AddLinearConstraint(
                P.A(),  # Coefficient matrix
                -np.inf * np.ones_like(P.b()),  # Lower bound
                P.b(),  # Upper bound
                x[self.q_dim*i:self.q_dim*(i+1)]
            )
        # 6. Inequality Constraints (Grasp hull)
        # for i in range(n_points - 2):
        #     prog.AddLinearConstraint(
        #         G.A(),
        #         -np.inf * np.ones_like(G.b()),
        #         G.b(),
        #         x[self.q_dim*(i+1):self.q_dim*(i+2)]
        #     )

        # 7. Joint limits (inequality)
        prog.AddBoundingBoxConstraint(
            np.tile(lower_joint_limits, n_points),
            np.tile(upper_joint_limits, n_points),
            x
        )

        # 7. Collision-free polytope constraints (MIP)
        # Ensure all points are contained in at least one polytope

        M = 1e6  # Big-M constant (adjust based on problem scale)
        for i in range(n_points):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]

            # Creates one bineary 0-1 variable for each polytope
            z_i = prog.NewBinaryVariables(len(c_free_polytopes), name=f"z_{i}")

            # Each point x_i must be contained in at least one polytope
            prog.AddLinearConstraint(sum(z_i) >= 1)

            for j, poly in enumerate(c_free_polytopes):
                A_j = poly.A()
                b_j = poly.b()
                z_j = z_i[j]

                # Add constraints for each row of the polytope
                for k in range(A_j.shape[0]):
                    # Construct coefficient matrix [A_j_row | M]
                    coeffs = np.hstack([A_j[k], M])
                    # Combine x_i and z_j into variable vector
                    vars = np.hstack([x_i, [z_j]])
                    # A_j @ x_i + M * z[j] <= b_j + M
                    prog.AddLinearConstraint(
                        coeffs,
                        -np.inf,
                        b_j[k] + M,
                        vars
                    )

        # 8. Connected components constraints
        connected_components = self._compute_connected_components(P, G, c_free_polytopes)
        K = len(connected_components)

        # create binary variables for component membership (excluding init ang goal)
        z = {}
        for i in range (1, n_points-1): # exclude init and goal
            z[i] = prog.NewBinaryVariables(K, f"z_{i}")
            # Each point must belong to exactly one component
            prog.AddLinearConstraint(sum(z[i]) == 1)

        # Consecutive points must belong to the same component, ignoring init and goal
        for i in range(1, n_points-2, 2):
            for k in range(K):
                prog.AddLinearConstraint(z[i][k] == z[i+1][k])

        # Enforce that each component contains at least one point
        for i in range(1, n_points-1):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]
            z_i = z[i]

            for k, comp in enumerate(connected_components):
                A_k = comp.A()
                b_k = comp.b()
                z_k = z_i[k]

                for j in range(A_k.shape[0]):
                    coeffs = np.hstack([A_k[j], M])
                    vars = np.hstack([x_i, [z_k]])
                    prog.AddLinearConstraint(coeffs, -np.inf, b_k[j] + M, vars)

        # Solve the problem
        solver = MosekSolver()
        result = solver.Solve(prog)

        if not result.is_success():
            return None

        return result.GetSolution(x).reshape(-1, self.q_dim)

    @staticmethod
    def _to_quadratic_form(C, c):
        return np.dot(C.T, C), -np.dot(C.T, c).flatten()

    def _compute_connected_components(
        self,
        placement: HPolyhedron,
        grasp: HPolyhedron,
        c_free: list[HPolyhedron]
    ) -> list[HPolyhedron]:
        """
        Computes convex connected components as:
        {placement ∩ grasp ∩ (union of c_free)}.
        Returns convex hulls of connected regions.
        """
        components = []

        # Compute intersections with each c_free polytope
        for poly in c_free:
            intersection = placement.Intersection(grasp).Intersection(poly)
            if not intersection.IsEmpty():
                components.append(intersection)

        # Merge overlapping components
        merged = []
        for comp in components:
            add_to_merged = False
            for m in merged:
                if comp.Intersection(m).IsEmpty():
                    continue
                # Merge overlapping components
                vertices_comp = visualizer.get_polytope_vertices(comp)
                vertices_m = visualizer.get_polytope_vertices(m)
                vertices = np.vstack([vertices_comp, vertices_m])
                merged_hull = ConvexHull(vertices)
                merged_poly = convex_hull_to_hpolyhedron(merged_hull)
                merged.remove(m)
                merged.append(merged_poly)
                add_to_merged = True
                break
            if not add_to_merged:
                merged.append(comp)

        return merged



    def _generate_cfree(self, irisalg: str = "irisnp") -> list[HPolyhedron]:
        if irisalg not in ["irisnp", "irisnp2", "iriszo"]:
            raise ValueError(f"Invalid irisalg: {irisalg}. Must be one of ['irisnp', 'irisnp2', 'iriszo']")

        generator = RandomGenerator(1234)
        checker = SceneGraphCollisionChecker(
            model=self.diagram,
            robot_model_instances=self.model_instances,
            edge_step_size=0.01,
        )

        parallelism = Parallelism(32)

        if irisalg != "irisnp":
            # For iriszo and irisnp2, we need to set the sampled iris options
            common_sampled_iris_options = CommonSampledIrisOptions()
            common_sampled_iris_options.delta = 0.05
            common_sampled_iris_options.epsilon = 0.01
            common_sampled_iris_options.max_iterations = 10
            common_sampled_iris_options.max_iterations_separating_planes = 50
            # common_sampled_iris_options.mixing_steps =
            common_sampled_iris_options.num_particles = 100
            common_sampled_iris_options.parallelism = parallelism
            common_sampled_iris_options.verbose = True
            common_sampled_iris_options.configuration_space_margin = 1e-4
            common_sampled_iris_options.termination_threshold = 1e-5
            common_sampled_iris_options.relative_termination_threshold = 1e-4
            common_sampled_iris_options.remove_all_collisions_possible = False

        options = IrisFromCliqueCoverOptions()
        options.num_points_per_visibility_round = 200
        options.coverage_termination_threshold = 0.99
        options.parallelism = parallelism

        if irisalg == "iriszo":
            iris_zo_options = IrisZoOptions()
            iris_zo_options.bisection_steps = 30
            iris_zo_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_zo_options
        elif irisalg == "irisnp2":
            iris_np2_options = IrisNp2Options()
            iris_np2_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_np2_options
        else:
            # irisnp uses the default IrisOptions object on options.iris_options.
            options.iris_options.configuration_space_margin = 1e-5

        # See https://github.com/RobotLocomotion/drake/issues/21343
        regions = IrisInConfigurationSpaceFromCliqueCover(checker, options, generator, [])
        print("\nnumber of regions: ", len(regions))
        return regions

## 4. Generate C-free (native Drake IRIS-ZO) and plan a trajectory

**Caveats, read before running:**
- `cs_free=None, irisalg="iriszo"` makes `__init__` call the class's own
  `_generate_cfree(irisalg="iriszo")` - clique-cover to 98% coverage over
  the full 10-dim configuration space, `Parallelism(32)`. This is the
  expensive step (minutes+, depends on hardware) and has not been run as
  part of writing this notebook.
- `compute_trajectory`'s MIQP step needs a MOSEK license (`MosekSolver`).
- `q0` (and `q_goal` below) were only checked against the grasp constraint
  (`h_grasp_eq`/`g_grasp_ineq`) on the *obstacle-free* standalone plant from
  section 2 - they have not been checked for collision against this
  notebook's actual `obstacle1`/`obstacle3`. IRIS-ZO will simply fail to
  find a region containing a point that's in collision; this hasn't been
  verified either way yet.

In [58]:
# Optionally load a precomputed set of collision-free polytopes from a file
from pydrake.geometry.optimization import LoadIrisRegionsYamlFile

cfree_regions_dict = LoadIrisRegionsYamlFile("data/cfree/cfree_full_98coverage.yaml")
cfree_regions = list(cfree_regions_dict.values())


In [59]:
# Check intersections between grasp, and collision-free polytopes


grasp_intersections = []
for i, poly in enumerate(cfree_regions):
    intersection = grasp_polytope.Intersection(poly)
    if not intersection.IsEmpty():
        print(f"Grasp polytope intersects with cfree region {i}")
        grasp_intersections.append(intersection)
    else:
        print(f"Grasp polytope does NOT intersect with cfree region {i}")
        
        
# Check on intersections between grasping polytopes that intersect with c-free:, and merge them into connected components
connected_components = [] # each item here is a component. Each component is a list of items from grasp_intersections 

for i, poly in enumerate(grasp_intersections):
    added_to_component = False
    for component in connected_components:
        for comp_poly in component:
            if not poly.Intersection(comp_poly).IsEmpty():
                component.append(poly)
                added_to_component = True
                break
        if added_to_component:
            break
    if not added_to_component:
        connected_components.append([poly])
        
        
print(f"Number of connected components: {len(connected_components)}")

print("Connected components:")
for i, component in enumerate(connected_components):
    print(f"Component {i}: {len(component)} polytopes")

Grasp polytope intersects with cfree region 0
Grasp polytope intersects with cfree region 1
Grasp polytope intersects with cfree region 2
Grasp polytope intersects with cfree region 3
Grasp polytope intersects with cfree region 4
Grasp polytope intersects with cfree region 5
Grasp polytope does NOT intersect with cfree region 6
Grasp polytope does NOT intersect with cfree region 7
Grasp polytope does NOT intersect with cfree region 8
Grasp polytope intersects with cfree region 9
Grasp polytope intersects with cfree region 10
Grasp polytope intersects with cfree region 11
Grasp polytope does NOT intersect with cfree region 12
Grasp polytope does NOT intersect with cfree region 13
Grasp polytope intersects with cfree region 14
Grasp polytope does NOT intersect with cfree region 15
Grasp polytope intersects with cfree region 16
Grasp polytope intersects with cfree region 17
Number of connected components: 1
Connected components:
Component 0: 12 polytopes


In [ ]:
model_instances = [
    plant.GetModelInstanceByName("panda"),
    plant.GetModelInstanceByName("panda_hand"),
    plant.GetModelInstanceByName("bottle_cap"),
]

planner = ManipulationPlanner(
    visualizer,
    placement_polytope,
    grasp_polytope,
    max_grasps=50,
    model_instances=model_instances,
    gripper_dim=4,
    # cs_free=cfree_regions,
    irisalg="irisnp",
)

print(f"Generated {len(planner.cs_free)} C-free regions.")

In [ ]:
from pydrake.geometry.optimization import SaveIrisRegionsYamlFile

SaveIrisRegionsYamlFile(
    "data/cfree/cfree_99coverage_irisnp.yaml",
    {
        f"region_{i:03d}": region
        for i, region in enumerate(planner.cs_free)
    },
)

# np.save(
#     "path_drake_1_49.npy",
#     path,
# )

### Plan a trajectory

`x_goal` reuses `q0` with only the cap angle advanced by `-1.0` rad (roughly
the wrist's own achievable range) - everything else about the pose is
unchanged, since the grasp constraint is exactly insensitive to the cap's
own value (finding 2 in `tc_space_investigation_summary.md` Part 2). This
keeps `x_goal` inside both `grasp_polytope` and `placement_polytope` by
construction, and is achievable within a single grasp cycle, so a sensible
first run should find a short path (small `n_grasps`).

In [ ]:
q_init = np.zeros(plant.num_positions())
q_init[idx_cap] = -0.0

q_goal = np.zeros(plant.num_positions())
q_goal[idx_cap] = 0.5

print(f"x_init  = {np.round(q_init, 4)}")
print(f"x_goal  = {np.round(q_goal, 4)}")

path, traj = planner.compute_trajectory(q_init, q_goal, display=True)

In [ ]:
planner.plant.SetPositions(planner.plant_context, q_init)
traj_path = planner.display_trajectory(traj, plotly=False)